# Augplot: a chart, then one sentence to improve it

Two examples: highlight the best CV models, then overlay supplied forecasts and intervals on observed data. **Modeling happens upstream; Augplot visualizes the results.** All data here is synthetic.

**Setup:** install using the [README](../README.md#quick-start), select your Augplot kernel, and run the first cell. It selects `openai/gpt-5.6-terra` and asks for your API key with hidden input. Static plots display in Retina quality automatically.

First run: four generations, plus one if you enable Plotly; each may need one repair request. Identical reruns use saved code. Your provider receives a data profile; generated Python runs locally and is not sandboxed.

In [ ]:
import os
from getpass import getpass

os.environ["AUGPLOT_MODEL"] = "openai/gpt-5.6-terra"
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

In [ ]:
import numpy as np
import pandas as pd

from augplot import Visualizer, plot

## 1. Compare models across two metrics

Five classifiers, the same five CV folds, two metrics where higher is better. Gradient boosting leads on mean accuracy; random forest leads on mean ROC-AUC. The neural network is less consistent across folds.

Start with ordinary grouped bars so the next refinement has a clear before/after.

In [ ]:
results_dict = {
    "Logistic regression": {
        "accuracy": np.array([0.80, 0.82, 0.81, 0.79, 0.84]),
        "roc_auc": np.array([0.88, 0.90, 0.89, 0.86, 0.90]),
    },
    "Random forest": {
        "accuracy": np.array([0.88, 0.90, 0.89, 0.86, 0.90]),
        "roc_auc": np.array([0.95, 0.96, 0.94, 0.93, 0.95]),
    },
    "Gradient boosting": {
        "accuracy": np.array([0.92, 0.93, 0.91, 0.92, 0.93]),
        "roc_auc": np.array([0.94, 0.95, 0.93, 0.92, 0.95]),
    },
    "Extra trees": {
        "accuracy": np.array([0.86, 0.88, 0.89, 0.87, 0.88]),
        "roc_auc": np.array([0.91, 0.93, 0.92, 0.90, 0.92]),
    },
    "Neural network": {
        "accuracy": np.array([0.94, 0.88, 0.93, 0.84, 0.95]),
        "roc_auc": np.array([0.96, 0.90, 0.95, 0.86, 0.97]),
    },
}

viz = plot(
    results_dict,
    prompt="Compare models in grouped bars: mean accuracy and ROC-AUC, with ±1 sample "
    "standard deviation across CV folds (not confidence intervals). Both metrics are "
    "higher-is-better. Use blue for accuracy and orange for ROC-AUC, readable model "
    "labels, a zero baseline, and room above the error bars. Label the metric colors.",
    backend="matplotlib",
)
# To let Augplot choose the chart instead: viz = plot(results_dict)

In [ ]:
print(viz.explanation)
print("Reused saved code:", viz.cache_hit)

## 2. “Highlight the best model”

One sentence replaces the manual work of finding winners, recoloring individual bars, adding hatching, positioning labels, and updating the legend. “Best” means highest observed mean for each metric, not statistical significance.

In [ ]:
viz.refine(
    "Highlight the best model for each metric: use a saturated color and diagonal "
    "hatching, fade the other bars, and label each winning mean to 3 decimals above "
    "its error bar. Keep the error bars and explain the highlight in the legend."
)

In [ ]:
# The generated function and its saved version are available for inspection.
print("Saved source:", viz.history_path)
# print(viz.code)

## 3. New results, same function

These synthetic updated scores make the neural network the accuracy winner. `render()` should move the highlight automatically, without calling the LLM. Export the current function under a readable name when you're happy with it.

In [ ]:
updated_results = {
    model: {metric: values.copy() for metric, values in metrics.items()}
    for model, metrics in results_dict.items()
}
updated_results["Neural network"]["accuracy"] = np.array([0.947, 0.943, 0.952, 0.938, 0.950])
viz.render(updated_results, title="Updated CV results — a new accuracy winner")

In [ ]:
from pathlib import Path

# Choose a fresh filename on reruns so we do not overwrite existing code.
export_path = Path("vis_utils.py")
version = 2
while export_path.exists():
    export_path = Path(f"vis_utils_{version}.py")
    version += 1

viz.save(export_path, function_name="plot_cv_results")

In [ ]:
# The printed snippet works for normal imports. Here we load the chosen file
# directly so this cell also works when a rerun selected a numbered filename.
import importlib.util

spec = importlib.util.spec_from_file_location(export_path.stem, export_path)
vis_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vis_utils)

fig = vis_utils.plot_cv_results(updated_results, figsize=(11, 5))
fig

## 4. Bring observations and model outputs

Start with a dot plot of weekly orders. The table also contains **supplied** forecasts and interval bounds, as an upstream model might export. The first six forecast weeks have actuals for comparison; the last two have not been observed.

These fixed synthetic predictions and bounds illustrate the display, not model accuracy or calibrated coverage. Augplot does not fit a model, generate predictions, or estimate intervals.

In [ ]:
rng = np.random.default_rng(42)
weeks = pd.date_range("2025-01-06", periods=32, freq="W-MON")
history = pd.DataFrame({
    "week": weeks[:24],
    "actual": np.round(450 + 9 * np.arange(24) + rng.normal(0, 30, 24)),
    "forecast": np.nan,
    "lower": np.nan,
    "upper": np.nan,
})
# Fixed stand-ins for predictions/intervals from an upstream model. No training here.
forecast_window = pd.DataFrame({
    "week": weeks[24:],
    "actual": [675, 710, 620, 713, 810, 728, np.nan, np.nan],
    "forecast": [668, 691, 679, 708, 734, 720, 755, 773],
    "lower": [628, 648, 634, 660, 684, 668, 699, 715],
    "upper": [708, 734, 724, 756, 784, 772, 811, 831],
})
forecast_results = pd.concat([history, forecast_window], ignore_index=True)

orders_viz = Visualizer(backend="seaborn")
orders_viz.fit(
    forecast_results,
    prompt="Show actual weekly orders as teal dots with readable date labels, "
    "a white background, and a light grid. Label the axes Week and Orders. "
    "For now show only non-missing actuals; leave forecast, lower, and upper hidden.",
)

## 5. “Show the forecast and where it missed”

Refine the same chart to overlay the supplied predictions and bounds. Two observed values fall outside the supplied interval; highlight them. Missing actuals stay blank, and interval coverage is deliberately unspecified.

In [ ]:
orders_viz.refine(
    "Overlay the supplied forecast as a dashed orange line and shade the supplied "
    "lower–upper interval. Mark the first forecast week, and highlight actuals outside "
    "the interval with red diamonds. Keep other actuals teal and missing actuals blank. "
    "Label the band 'Supplied interval (synthetic)'; do not infer a confidence level "
    "or recompute any predictions or bounds."
)

## 6. Optional: inspect the supplied forecast interactively

Set the toggle to `True` for a Plotly version with dates, actuals, predictions, and interval bounds on hover.

In [ ]:
RUN_PLOTLY = False

if RUN_PLOTLY:
    interactive_viz = plot(
        forecast_results,
        prompt="Plot actual weekly orders as teal dots, the supplied forecast as a dashed "
        "orange line, and the supplied lower–upper interval as a translucent orange band. "
        "Highlight actuals outside the interval with red diamonds. Mark the first forecast "
        "week. Hover should show date, actual, forecast, lower, and upper. Leave missing "
        "values blank, use a white background, and label the band 'Supplied interval "
        "(synthetic)'. Do not fit a model or recompute predictions or intervals.",
        backend="plotly",
    )

**Rerunning:** run from the original `plot()` cell to replay the same refinement. Repeating only `refine()` edits the current version again. Keep `.augplot/` with this notebook. [How history works](../docs/visualization-history.md).